# Gemma 3 270M — V2 optimized one-minute Colab notebook

**One notebook. One v5e-1 TPU. One fixed 60-second scratch-training run.**

1. Upload this notebook to Google Colab Pro.
2. Select **Runtime → Change runtime type → TPU v5e-1**.
3. Select **Runtime → Run all**. Authorize Google Drive and supply your Hugging Face token when requested.

Accept the [Gemma repository terms](https://huggingface.co/google/gemma-3-270m) first. A Colab Secret named `HF_TOKEN` is supported. No source files need uploading and no configuration edits are required.

V2 combines intact-chunk document packing, batch size **2**, resident TPU inputs, compiled groups of up to **16 updates**, buffered metrics, and reusable data/environment/compiler caches. Weights start randomly with seed **42**. AdamW and the exact full-vocabulary loss use the known V1 learning configuration.

Results go to **My Drive / ShiningGem / colab-v5e1 / V2 / unique run folder**. The full resumable checkpoint is approximately **3.2 GB**. Keep Colab connected until saving finishes.

The timer covers **training only**. Setup, compilation, validation, caching, and checkpoint saving consume additional runtime. Compute units are estimated at your supplied **2.92 CU/hour**; 60 seconds of training is approximately **0.0487 CU**. Colab billing is authoritative.

**Local checks and tests have not been run, as requested. V2 TPU throughput and learning improvement are unmeasured until you run this notebook.**


## Fixed configuration and requirements

| Setting | Fixed value |
|---|---:|
| Device | One TPU v5e-1 |
| Architecture | Native `gm.nn.Gemma3_270M`, 268,098,176 parameters |
| Vocabulary | 262,144; tied embedding/output matrix |
| Initial weights | Random; seed 42 |
| Training / validation documents | 10,000 / 32, identical to V1 by ordered document hashes |
| Validation targets | 13,229 |
| Sequence / batch | 1,024 positions / 2 packed sequences |
| Main dispatch | Up to 16 updates; one update near the deadline |
| Compute / master weights | BF16 / FP32 |
| Optimizer | AdamW, LR 0.0003, warmup 5 updates, clip 1.0 |
| Adam b1 / b2 / epsilon | 0.9 / 0.999 / 1e-8 |
| Weight decay | 0.01 on matrix/tensor weights |
| Training interval | 60 seconds nominal; actual completed-work time reported |

| ID | Requirement | Verification in your Colab run |
|---|---|---|
| V2-001 | The runner shall execute one training attempt per run folder. | Exclusive start marker; repeated training-cell execution stops. |
| V2-002 | The runner shall use exactly one v5e TPU. | Device platform, kind, and count. |
| V2-003 | The packer shall preserve every original target once per dataset pass. | Source chunk assignments, target equality, and target totals. |
| V2-004 | The model shall isolate attention between packed segments. | Causal segment mask and an explicit mask verification before training. |
| V2-005 | The packer shall preserve the original context and positions of every chunk. | Whole-chunk packing and position equality. |
| V2-006 | The runner shall place training inputs on the TPU before starting the timer. | Device ownership and completed transfer barrier. |
| V2-007 | The runner shall limit each compiled call to 16 optimizer updates. | Completed dispatch counts. |
| V2-008 | The runner shall cease dispatching training calls at 60 seconds. | Monotonic timestamps, completed-work synchronization, and overrun report. |
| V2-009 | The runner shall reject an update with nonfinite loss or gradient. | Device-side finite checks; stop without retry. |
| V2-010 | The evaluator shall score the same V1 validation documents before and after training. | Document hashes and 13,229 target count. |
| V2-011 | The runner shall persist one full resumable checkpoint. | Local/persistent SHA256 equality, array count, and state schema. |
| V2-012 | The report shall provide separate training and overhead timings. | Phase durations and compute-unit estimates. |
| V2-013 | The model shall retain the native architecture and tied embedding/output table. | Parameter count, table shape, and unique table count. |
| V2-014 | The runner shall perform zero optimizer updates outside the training interval. | Zero initial step; final step equals timed update count. |
| V2-015 | Training shall change a transformer parameter. | Initial/final parameter probe. |
| V2-016 | The runner shall complete at least five optimizer updates. | Completed update count. |

Requirements use NASA-style numbered **shall** statements. Engineering status, learning status, and performance goals are reported separately.

| Performance goal | Measured V1 | V2 goal, not a guarantee |
|---|---:|---:|
| Useful position occupancy | 52.6% | At least 95% |
| Useful tokens/second | 11,406 | At least 18,000 |
| Useful tokens in 60 seconds | 684,589 | At least 1,080,000 |
| Final validation loss | 6.872893 | At most 6.872893 |

The comparison uses your saved V1 measurement; this notebook does not rerun V1. A higher token rate does not automatically imply lower validation loss.


## 1. Create the run and prepare the pinned environment
The embedded V1 dependency lock is reused. The Colab kernel stays separate from the managed Python 3.12 environment.


In [ ]:
import os
import hashlib
import sys
import json
import time
import uuid
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_START_MONOTONIC = time.monotonic()
BASE = Path("/content/gemma_colab_pilot")
BASE.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
WORK = BASE / RUN_ID
WORK.mkdir()
if shutil.disk_usage(BASE).free < 8 * 1024**3:
    raise RuntimeError("This full-checkpoint pilot needs at least 8 GiB of free local disk.")

DEPENDENCY_LOCK = r'''
# This file was autogenerated by uv via the following command:
#    uv pip compile /tmp/gemma-colab-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_35 -o /tmp/gemma-colab-requirements.lock
absl-py==2.5.1
    # via
    #   array-record
    #   chex
    #   clu
    #   dm-tree
    #   etils
    #   gemma
    #   grain
    #   kauldron
    #   keras
    #   ml-collections
    #   optax
    #   orbax-checkpoint
    #   tensorboard
    #   tensorflow
    #   tensorflow-datasets
    #   tensorflow-metadata
    #   xmanager
aiofile==3.12.3
    # via py-key-value-aio
aiofiles==25.1.0
    # via orbax-checkpoint
aiohappyeyeballs==2.7.1
    # via aiohttp
aiohttp==3.14.5
    # via
    #   aiohttp-retry
    #   cloud-sql-python-connector
    #   fsspec
    #   kubernetes
aiohttp-retry==2.9.1
    # via kubernetes
aiosignal==1.4.0
    # via aiohttp
alembic==1.20.0
    # via xmanager
altair==6.3.0
    # via kauldron
annotated-types==0.8.0
    # via pydantic
anyio==4.15.1
    # via
    #   google-genai
    #   httpx
    #   httpx2
    #   mcp
    #   py-key-value-aio
    #   sse-starlette
    #   starlette
    #   watchfiles
anywidget==0.11.0
    # via dialog
array-record==0.8.6
    # via
    #   grain
    #   tensorflow-datasets
asttokens==3.0.2
    # via stack-data
astunparse==1.6.3
    # via tensorflow
async-generator==1.10
    # via xmanager
attrs==26.1.0
    # via
    #   aiohttp
    #   cyclopts
    #   dm-tree
    #   jsonschema
    #   jsonschema-path
    #   referencing
    #   xmanager
authlib==1.9.1
    # via fastmcp-slim
beartype==0.23.1
    # via py-key-value-aio
cachetools==7.2.4
    # via py-key-value-aio
caio==0.12.9
    # via aiofile
certifi==2026.10.19
    # via
    #   fastmcp-slim
    #   google-cloud-aiplatform
    #   httpcore
    #   httpx
    #   kubernetes
    #   requests
cffi==2.1.1
    # via cryptography
charset-normalizer==3.5.2
    # via requests
chex==0.1.92
    # via kauldron
click==8.5.0
    # via
    #   huggingface-hub
    #   uvicorn
cloud-sql-python-connector==1.23.0
    # via xmanager
cloudpickle==3.1.2
    # via
    #   grain
    #   joblib
clu==0.0.12
    # via kauldron
comm==0.2.3
    # via ipywidgets
contourpy==1.4.0
    # via matplotlib
cryptography==50.0.2
    # via
    #   authlib
    #   cloud-sql-python-connector
    #   google-auth
    #   joserfc
    #   pyjwt
    #   secretstorage
cycler==0.12.1
    # via matplotlib
cyclopts==5.2.0
    # via fastmcp-slim
datasets==5.1.0
    # via -r /tmp/gemma-colab-requirements.in
dialog==1.1.0
    # via gemma
dill==0.4.1
    # via
    #   datasets
    #   multiprocess
distro==1.9.0
    # via google-genai
dm-tree==0.1.10
    # via tensorflow-datasets
dnspython==2.9.0
    # via
    #   cloud-sql-python-connector
    #   email-validator
docker==7.2.0
    # via xmanager
docstring-parser==0.18.0
    # via
    #   cyclopts
    #   google-cloud-aiplatform
    #   simple-parsing
durationpy==0.11
    # via kubernetes
einops==0.8.2
    # via
    #   etils
    #   gemma
    #   kauldron
email-validator==2.3.0
    # via pydantic
etils==1.14.0
    # via
    #   array-record
    #   clu
    #   dialog
    #   gemma
    #   grain
    #   kauldron
    #   orbax-checkpoint
    #   tensorflow-datasets
    #   xmanager
exceptiongroup==1.3.1
    # via fastmcp-slim
executing==2.3.0
    # via stack-data
fastmcp==4.1.3
    # via gemma
fastmcp-slim==4.1.3
    # via fastmcp
filelock==4.2.4
    # via
    #   datasets
    #   huggingface-hub
flatbuffers==25.12.19
    # via tensorflow
flax==0.12.10
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   clu
    #   gemma
    #   kauldron
fonttools==4.67.0
    # via matplotlib
frozenlist==1.8.0
    # via
    #   aiohttp
    #   aiosignal
fsspec==2026.7.0
    # via
    #   datasets
    #   etils
    #   huggingface-hub
gast==0.7.0
    # via tensorflow
gemma==4.0.1
    # via -r /tmp/gemma-colab-requirements.in
gitdb==4.0.12
    # via gitpython
gitpython==3.2.1
    # via xmanager
google-api-core==2.42.0
    # via
    #   google-api-python-client
    #   google-cloud-aiplatform
    #   google-cloud-bigquery
    #   google-cloud-core
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   google-cloud-storage
    #   xmanager
google-api-python-client==2.201.0
    # via xmanager
google-auth==2.62.0
    # via
    #   cloud-sql-python-connector
    #   google-api-core
    #   google-api-python-client
    #   google-auth-httplib2
    #   google-cloud-aiplatform
    #   google-cloud-bigquery
    #   google-cloud-core
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   google-cloud-storage
    #   google-genai
    #   xmanager
google-auth-httplib2==0.4.4
    # via google-api-python-client
google-cloud-aiplatform==2.6.0
    # via xmanager
google-cloud-bigquery==3.46.2
    # via google-cloud-aiplatform
google-cloud-core==2.8.0
    # via
    #   google-cloud-bigquery
    #   google-cloud-storage
google-cloud-resource-manager==1.19.0
    # via google-cloud-aiplatform
google-cloud-sql==0.1.1
    # via cloud-sql-python-connector
google-cloud-storage==3.17.1
    # via
    #   google-cloud-aiplatform
    #   xmanager
google-crc32c==1.9.0
    # via
    #   google-cloud-storage
    #   google-resumable-media
google-genai==2.32.0
    # via google-cloud-aiplatform
google-pasta==0.2.0
    # via tensorflow
google-resumable-media==2.11.0
    # via
    #   google-cloud-bigquery
    #   google-cloud-storage
googleapis-common-protos==1.75.5
    # via
    #   google-api-core
    #   grpc-google-iam-v1
    #   grpcio-status
    #   tensorflow-metadata
    #   xmanager
grain==0.2.18
    # via
    #   gemma
    #   kauldron
graphviz==0.21
    # via kauldron
griffelib==2.4.0
    # via fastmcp-slim
grpc-google-iam-v1==0.14.5
    # via google-cloud-resource-manager
grpcio==1.84.0
    # via
    #   google-api-core
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   googleapis-common-protos
    #   grpc-google-iam-v1
    #   grpcio-status
    #   tensorboard
    #   tensorflow
    #   xmanager
grpcio-status==1.84.0
    # via google-api-core
h11==0.16.0
    # via
    #   httpcore
    #   httpcore2
    #   uvicorn
h5py==3.14.0
    # via
    #   keras
    #   tensorflow
hf-xet==1.7.0
    # via huggingface-hub
httpcore==1.0.9
    # via httpx
httpcore2==2.16.0
    # via httpx2
httplib2==0.32.0
    # via
    #   google-api-python-client
    #   google-auth-httplib2
httpx==0.28.1
    # via google-genai
httpx2==2.16.0
    # via
    #   fastmcp-slim
    #   huggingface-hub
    #   mcp
huggingface-hub==2.2.0
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   datasets
    #   tokenizers
humanize==4.16.0
    # via
    #   orbax-checkpoint
    #   xmanager
idna==3.20
    # via
    #   anyio
    #   email-validator
    #   httpx
    #   httpx2
    #   requests
    #   yarl
imageio==2.38.1
    # via scikit-image
immutabledict==4.3.1
    # via
    #   kauldron
    #   tensorflow-datasets
    #   xmanager
ipython==9.17.1
    # via
    #   ipywidgets
    #   mediapy
ipython-pygments-lexers==1.1.1
    # via ipython
ipywidgets==8.1.9
    # via anywidget
jaraco-classes==3.4.0
    # via keyring
jaraco-context==6.1.2
    # via keyring
jaraco-functools==4.6.0
    # via keyring
jax==0.12.0
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   chex
    #   clu
    #   flax
    #   gemma
    #   kauldron
    #   optax
    #   orbax-checkpoint
jaxlib==0.12.0
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   chex
    #   clu
    #   jax
    #   optax
jaxtyping==0.3.11
    # via
    #   gemma
    #   kauldron
jedi==0.20.1
    # via ipython
jeepney==0.9.0
    # via
    #   keyring
    #   secretstorage
jinja2==3.1.6
    # via altair
joblib==1.6.0
    # via scikit-learn
joserfc==1.7.7
    # via
    #   authlib
    #   fastmcp-slim
jsonref==1.1.0
    # via fastmcp-slim
jsonschema==4.26.0
    # via
    #   altair
    #   mcp
jsonschema-path==0.5.0
    # via fastmcp-slim
jsonschema-specifications==2025.9.1
    # via jsonschema
jupyterlab-widgets==3.0.17
    # via ipywidgets
kauldron==1.4.2
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   gemma
keras==3.15.1
    # via tensorflow
keyring==25.7.0
    # via py-key-value-aio
kiwisolver==1.5.1
    # via matplotlib
kubernetes==37.0.1
    # via xmanager
lark==1.3.1
    # via
    #   dialog
    #   kauldron
lazy-loader==0.6
    # via scikit-image
libclang==18.1.1
    # via tensorflow
libtpu==0.0.50
    # via jax
mako==1.4.3
    # via alembic
markdown==3.11.2
    # via tensorboard
markdown-it-py==4.2.0
    # via rich
markupsafe==3.0.4
    # via
    #   jinja2
    #   mako
    #   werkzeug
matplotlib==3.11.2
    # via mediapy
matplotlib-inline==0.2.2
    # via ipython
mcp==2.3.0
    # via
    #   dialog
    #   fastmcp-slim
    #   gemma
mcp-types==2.3.0
    # via
    #   fastmcp-slim
    #   mcp
mdurl==0.1.2
    # via markdown-it-py
mediapy==1.2.7
    # via kauldron
ml-collections==1.1.0
    # via
    #   clu
    #   kauldron
ml-dtypes==0.6.0
    # via
    #   jax
    #   jaxlib
    #   keras
    #   tensorflow
    #   tensorstore
more-itertools==11.2.1
    # via
    #   jaraco-classes
    #   jaraco-functools
msgpack==1.2.3
    # via
    #   flax
    #   orbax-checkpoint
multidict==7.1.0
    # via
    #   aiohttp
    #   yarl
multiprocess==0.70.19
    # via datasets
namex==0.1.0
    # via keras
narwhals==2.27.1
    # via
    #   altair
    #   scikit-learn
networkx==3.7
    # via scikit-image
numpy==2.5.4
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   chex
    #   clu
    #   contourpy
    #   datasets
    #   dialog
    #   dm-tree
    #   etils
    #   flax
    #   gemma
    #   grain
    #   h5py
    #   imageio
    #   jax
    #   jaxlib
    #   kauldron
    #   keras
    #   matplotlib
    #   mediapy
    #   ml-dtypes
    #   opencv-python
    #   optax
    #   orbax-checkpoint
    #   pandas
    #   scikit-image
    #   scikit-learn
    #   scipy
    #   tensorboard
    #   tensorflow
    #   tensorflow-datasets
    #   tensorstore
    #   tifffile
    #   treescope
oauthlib==4.0.0
    # via requests-oauthlib
openapi-pydantic==0.6.0
    # via fastmcp-slim
opencv-python==5.0.0.93
    # via kauldron
opentelemetry-api==1.45.1
    # via
    #   fastmcp-slim
    #   google-api-core
    #   mcp
opt-einsum==3.4.0
    # via
    #   jax
    #   tensorflow
optax==0.2.8
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   flax
    #   kauldron
optree==0.20.0
    # via keras
orbax-checkpoint==0.12.8
    # via
    #   flax
    #   gemma
    #   kauldron
packaging==26.3
    # via
    #   altair
    #   clu
    #   datasets
    #   fastmcp-slim
    #   google-cloud-aiplatform
    #   google-cloud-bigquery
    #   huggingface-hub
    #   keras
    #   lazy-loader
    #   matplotlib
    #   scikit-image
    #   tensorboard
    #   tensorflow
    #   wheel
pandas==3.0.6
    # via
    #   datasets
    #   kauldron
parso==0.8.7
    # via jedi
pathable==0.6.0
    # via jsonschema-path
pexpect==4.9.0
    # via ipython
pillow==13.0.0
    # via
    #   dialog
    #   imageio
    #   matplotlib
    #   mediapy
    #   scikit-image
    #   tensorboard
platformdirs==4.13.5
    # via fastmcp-slim
portpicker==1.6.0
    # via grain
prometheus-client==0.26.0
    # via orbax-checkpoint
promise==2.3
    # via tensorflow-datasets
prompt-toolkit==3.0.53
    # via ipython
propcache==0.5.4
    # via
    #   aiohttp
    #   yarl
proto-plus==1.29.0
    # via
    #   google-api-core
    #   google-cloud-aiplatform
    #   google-cloud-resource-manager
    #   google-cloud-sql
protobuf==6.33.6
    # via
    #   google-api-core
    #   google-cloud-aiplatform
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   googleapis-common-protos
    #   grain
    #   grpc-google-iam-v1
    #   grpcio-status
    #   orbax-checkpoint
    #   proto-plus
    #   tensorboard
    #   tensorflow
    #   tensorflow-datasets
    #   tensorflow-metadata
    #   xmanager
psutil==7.2.2
    # via
    #   ipython
    #   orbax-checkpoint
    #   portpicker
    #   tensorflow-datasets
    #   xmanager
psygnal==0.16.1
    # via anywidget
ptyprocess==0.7.0
    # via pexpect
pure-eval==0.2.4
    # via stack-data
py-key-value-aio==0.4.6
    # via fastmcp-slim
pyarrow==26.0.0
    # via
    #   datasets
    #   tensorflow-datasets
pyasn1==0.6.4
    # via pyasn1-modules
pyasn1-modules==0.4.2
    # via google-auth
pycparser==3.11
    # via cffi
pydantic==2.14.1
    # via
    #   fastmcp-slim
    #   google-cloud-aiplatform
    #   google-genai
    #   kubernetes
    #   mcp
    #   mcp-types
    #   openapi-pydantic
    #   pydantic-settings
pydantic-core==2.50.1
    # via pydantic
pydantic-settings==2.16.1
    # via fastmcp-slim
pydub==0.25.1
    # via dialog
pygments==2.21.0
    # via
    #   ipython
    #   ipython-pygments-lexers
    #   rich
    #   rich-rst
pyjwt==2.15.1
    # via mcp
pyparsing==3.3.3
    # via
    #   httplib2
    #   matplotlib
pyperclip==1.11.0
    # via fastmcp-slim
python-dateutil==2.9.0.post0
    # via
    #   google-cloud-bigquery
    #   kubernetes
    #   matplotlib
    #   pandas
python-dotenv==1.2.4
    # via
    #   fastmcp-slim
    #   pydantic-settings
    #   xmanager
python-multipart==0.0.32
    # via
    #   fastmcp-slim
    #   mcp
pyyaml==6.0.3
    # via
    #   datasets
    #   fastmcp-slim
    #   flax
    #   huggingface-hub
    #   jsonschema-path
    #   kubernetes
    #   ml-collections
    #   orbax-checkpoint
    #   xmanager
referencing==0.37.0
    # via
    #   jsonschema
    #   jsonschema-path
    #   jsonschema-specifications
requests==2.34.2
    # via
    #   cloud-sql-python-connector
    #   dialog
    #   docker
    #   google-api-core
    #   google-auth
    #   google-cloud-bigquery
    #   google-cloud-storage
    #   google-genai
    #   jax
    #   kubernetes
    #   requests-oauthlib
    #   tensorflow
    #   tensorflow-datasets
    #   xmanager
requests-oauthlib==2.0.0
    # via kubernetes
rich==15.0.0
    # via
    #   cyclopts
    #   fastmcp-slim
    #   flax
    #   keras
    #   rich-rst
rich-rst==2.2.0
    # via cyclopts
rpds-py==2026.9.1
    # via
    #   jsonschema
    #   referencing
scikit-image==0.26.0
    # via kauldron
scikit-learn==1.9.1
    # via kauldron
scipy==1.18.1
    # via
    #   jax
    #   jaxlib
    #   scikit-image
    #   scikit-learn
secretstorage==3.5.0
    # via keyring
sentencepiece==0.2.2
    # via gemma
setuptools==84.2.1
    # via
    #   tensorboard
    #   tensorflow
simple-parsing==0.1.9
    # via tensorflow-datasets
simplejson==4.2.0
    # via orbax-checkpoint
six==1.17.0
    # via
    #   astunparse
    #   google-pasta
    #   promise
    #   python-dateutil
    #   tensorflow
smmap==5.0.3
    # via gitdb
sniffio==1.3.1
    # via google-genai
sqlalchemy==2.1.5
    # via
    #   alembic
    #   xmanager
sqlparse==0.6.0
    # via xmanager
sse-starlette==3.5.0
    # via mcp
stack-data==0.6.3
    # via ipython
starlette==1.9.0
    # via
    #   fastmcp-slim
    #   mcp
    #   sse-starlette
tabulate==0.10.0
    # via
    #   kauldron
    #   xmanager
tenacity==9.1.4
    # via google-genai
tensorboard==2.21.0
    # via kauldron
tensorboard-data-server==0.7.2
    # via tensorboard
tensorflow==2.21.0
    # via kauldron
tensorflow-datasets==4.9.10
    # via kauldron
tensorflow-metadata==1.21.0
    # via tensorflow-datasets
tensorstore==0.1.85
    # via
    #   flax
    #   orbax-checkpoint
termcolor==3.3.0
    # via
    #   tensorflow
    #   tensorflow-datasets
    #   xmanager
threadpoolctl==3.7.0
    # via scikit-learn
tifffile==2026.10.15
    # via scikit-image
tokenizers==0.23.3
    # via -r /tmp/gemma-colab-requirements.in
toml==0.10.2
    # via tensorflow-datasets
toolz==1.2.0
    # via chex
tqdm==4.70.1
    # via
    #   datasets
    #   etils
    #   huggingface-hub
    #   kauldron
    #   tensorflow-datasets
traitlets==5.16.1
    # via
    #   ipython
    #   ipywidgets
    #   matplotlib-inline
treescope==0.1.10
    # via
    #   flax
    #   gemma
truststore==0.10.5
    # via
    #   httpcore2
    #   httpx2
typeguard==4.6.0
    # via kauldron
typing-extensions==4.16.0
    # via
    #   aiohttp
    #   aiosignal
    #   alembic
    #   altair
    #   anyio
    #   anywidget
    #   chex
    #   clu
    #   etils
    #   exceptiongroup
    #   fastmcp-slim
    #   flax
    #   google-cloud-aiplatform
    #   google-genai
    #   grpcio
    #   httpx2
    #   huggingface-hub
    #   kauldron
    #   kubernetes
    #   mcp
    #   mcp-types
    #   opentelemetry-api
    #   optree
    #   orbax-checkpoint
    #   py-key-value-aio
    #   pydantic
    #   pydantic-core
    #   pydantic-settings
    #   referencing
    #   simple-parsing
    #   sqlalchemy
    #   starlette
    #   tensorflow
    #   typeguard
    #   typing-inspection
typing-inspection==0.4.4
    # via
    #   mcp
    #   pydantic
    #   pydantic-settings
uncalled-for==0.4.1
    # via fastmcp-slim
uritemplate==4.2.0
    # via google-api-python-client
urllib3==2.8.0
    # via
    #   docker
    #   kubernetes
    #   requests
uvicorn==0.54.0
    # via
    #   fastmcp-slim
    #   mcp
uvloop==0.23.0
    # via orbax-checkpoint
wadler-lindig==0.1.7
    # via jaxtyping
watchfiles==1.3.0
    # via fastmcp-slim
wcwidth==0.9.3
    # via prompt-toolkit
websocket-client==1.9.2
    # via kubernetes
websockets==16.1.1
    # via
    #   fastmcp-slim
    #   google-genai
werkzeug==3.1.9
    # via tensorboard
wheel==0.48.0
    # via astunparse
widgetsnbextension==4.0.16
    # via ipywidgets
wrapt==2.5.1
    # via
    #   clu
    #   dm-tree
    #   tensorflow
    #   tensorflow-datasets
xmanager==0.8.4
    # via kauldron
xxhash==4.0.2
    # via datasets
yarl==1.25.1
    # via aiohttp
zipp==4.1.1
    # via etils

'''

LOCK_PATH = WORK / "requirements.lock"
LOCK_PATH.write_text(DEPENDENCY_LOCK)
TOOLING = BASE / ".bootstrap"
if not (TOOLING / "uv").is_dir():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(TOOLING), "uv==0.11.15"], check=True)
uv_env = dict(os.environ, PYTHONPATH=str(TOOLING))
uv = [sys.executable, "-m", "uv"]
PYTHON = BASE / ".venv" / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([*uv, "venv", "--python", "3.12", "--managed-python", str(PYTHON.parent.parent)], env=uv_env, check=True)
LOCK_DIGEST = hashlib.sha256(DEPENDENCY_LOCK.encode()).hexdigest()
ENV_MARKER = PYTHON.parent.parent / ".v2-lock-sha256"
if not ENV_MARKER.exists() or ENV_MARKER.read_text().strip() != LOCK_DIGEST:
    print("Installing the pinned Gemma/JAX environment...", flush=True)
    subprocess.run([*uv, "pip", "install", "--quiet", "--python", str(PYTHON), "-r", str(LOCK_PATH)], env=uv_env, check=True)
    ENV_MARKER.write_text(LOCK_DIGEST + "\n")
else:
    print("Reusing the matching installed environment.", flush=True)
SETUP_SECONDS = time.monotonic() - NOTEBOOK_START_MONOTONIC
print(f"Environment ready. Installation/setup: {SETUP_SECONDS:.1f} seconds.")


## 2. Connect Google Drive and authenticate
The token is passed only to data preparation and is excluded from reports and checkpoints.


In [ ]:
from google.colab import drive, userdata
from getpass import getpass

drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/ShiningGem/colab-v5e1/V2") / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
PERSISTENT_JAX_CACHE = OUTPUT.parent.parent / "V2-cache" / "jax"
if PERSISTENT_JAX_CACHE.exists():
    shutil.copytree(PERSISTENT_JAX_CACHE, BASE / "jax-cache-V2", dirs_exist_ok=True)
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token with access to google/gemma-3-270m: ")
if not HF_TOKEN.strip():
    raise ValueError("A Hugging Face token with Gemma repository access is required.")
print("V2 results folder:", OUTPUT)


## 3. Write the complete embedded V2 program
Everything used by the experiment is embedded in this notebook. The fixed revisions and V1 document hashes prevent a silent data change.


In [ ]:
PROGRAM_SOURCE = '"""Gemma Colab V2: one fixed 60-second scratch-training run on one v5e TPU.\nEmbedded verbatim in the self-contained notebook. CPU fixtures are development only.\n"""\nfrom __future__ import annotations\nimport argparse\nimport dataclasses\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\nimport traceback\nimport zipfile\nimport numpy as np\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\\n")\n    temporary.replace(path)\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef chunk_document(ids, sequence_length, bos_id, eos_id, pad_id=0):\n    """Every target appears once; consecutive chunks overlap by one input token."""\n    if pad_id in ids:\n        raise ValueError("Document encodes a padding ID; cannot silently mask real text.")\n    sequence = [bos_id, *ids, eos_id]\n    for start in range(0, len(sequence) - 1, sequence_length):\n        window = sequence[start:start + sequence_length + 1]\n        count = len(window) - 1\n        tokens = np.full(sequence_length, pad_id, np.int32)\n        targets = np.full(sequence_length, pad_id, np.int32)\n        mask = np.zeros(sequence_length, np.bool_)\n        tokens[:count] = window[:-1]\n        targets[:count] = window[1:]\n        mask[:count] = True\n        yield tokens, targets, mask\n\n\ndef prepare(config, destination):\n    from datasets import load_dataset\n    from huggingface_hub import HfApi, snapshot_download\n    from tokenizers import Tokenizer\n\n    started = time.monotonic()\n    destination = Path(destination)\n    if destination.exists():\n        raise FileExistsError(f"Refusing to replace prepared data: {destination}")\n    destination.mkdir(parents=True)\n    api = HfApi()\n    model_sha = api.model_info(config["model_id"], revision=config["model_revision"]).sha\n    data_sha = api.dataset_info(config["dataset_id"], revision=config["dataset_revision"]).sha\n    tokenizer_dir = destination / "tokenizer"\n    snapshot_download(config["model_id"], revision=model_sha, local_dir=tokenizer_dir,\n                      allow_patterns=["config.json", "tokenizer.json", "tokenizer_config.json",\n                                      "special_tokens_map.json", "tokenizer.model"])\n    tokenizer = Tokenizer.from_file(str(tokenizer_dir / "tokenizer.json"))\n    model_config = read_json(tokenizer_dir / "config.json")\n    text_config = model_config.get("text_config", model_config)\n    expected = {"hidden_size": 640, "intermediate_size": 2048, "num_hidden_layers": 18,\n                "num_attention_heads": 4, "num_key_value_heads": 1, "head_dim": 256,\n                "vocab_size": 262144}\n    for key, value in expected.items():\n        if text_config.get(key) != value:\n            raise ValueError(f"Unexpected native model setting {key}: {text_config.get(key)}")\n    bos = text_config["bos_token_id"]\n    eos = text_config["eos_token_id"]\n    if isinstance(eos, list):\n        eos = eos[0]\n    pad = text_config.get("pad_token_id", 0)\n    if pad != 0:\n        raise ValueError("Gemma library expects padding ID 0.")\n    stream = load_dataset(config["dataset_id"], config["dataset_config"], split="train",\n                          revision=data_sha, streaming=True)\n    selected = {"train": [], "validation": []}\n    seen = set()\n    desired = config["train_documents"] + config["validation_documents"]\n    scanned = 0\n    for document in stream:\n        scanned += 1\n        text = document["text"]\n        if not text.strip():\n            continue\n        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()\n        if digest in seen:\n            continue\n        seen.add(digest)\n        split = "validation" if len(selected["validation"]) < config["validation_documents"] else "train"\n        selected[split].append({"id": str(document.get("id", digest)), "sha256": digest, "text": text})\n        if len(seen) == desired:\n            break\n    if len(seen) != desired:\n        raise RuntimeError(f"Only found {len(seen)} unique nonempty documents; need {desired}.")\n    manifest = {"model_id": config["model_id"], "model_revision": model_sha,\n                "dataset_id": config["dataset_id"], "dataset_config": config["dataset_config"],\n                "dataset_revision": data_sha, "documents_scanned": scanned,\n                "sequence_length": config["sequence_length"], "special_ids": {"bos": bos, "eos": eos, "pad": pad},\n                "native_config": expected, "splits": {}, "files": {}}\n    for split, documents in selected.items():\n        rows, metadata, roundtrips = [], [], 0\n        with (destination / f"{split}.jsonl").open("w", encoding="utf-8") as output:\n            for document_index, document in enumerate(documents):\n                output.write(json.dumps(document, ensure_ascii=False) + "\\n")\n                ids = tokenizer.encode(document["text"], add_special_tokens=False).ids\n                roundtrips += tokenizer.decode(ids, skip_special_tokens=False) == document["text"]\n                if not ids or min(ids) < 0 or max(ids) >= expected["vocab_size"]:\n                    raise ValueError(f"Invalid token IDs for {split} document {document_index}.")\n                rows.extend((*chunk, document_index) for chunk in chunk_document(\n                    ids, config["sequence_length"], bos, eos, pad))\n                metadata.append({key: document[key] for key in ("id", "sha256")})\n        tokens, targets, masks, doc_indices = zip(*rows)\n        arrays = {"tokens": np.stack(tokens), "targets": np.stack(targets),\n                  "mask": np.stack(masks), "document_index": np.array(doc_indices, np.int32)}\n        np.savez_compressed(destination / f"{split}.npz", **arrays)\n        manifest["splits"][split] = {"documents": metadata, "blocks": len(rows),\n            "scored_tokens": int(arrays["mask"].sum()), "exact_roundtrip_documents": roundtrips,\n            "original_utf8_bytes": sum(len(doc["text"].encode("utf-8")) for doc in documents)}\n        print(f"{split}: {len(documents)} documents, {len(rows)} blocks", flush=True)\n    if manifest["splits"]["train"]["blocks"] < config["expected_devices"]:\n        raise ValueError("Need at least one distinct training block per chip.")\n    for path in sorted(destination.rglob("*")):\n        if path.is_file() and ".cache" not in path.parts:\n            manifest["files"][str(path.relative_to(destination))] = sha256_file(path)\n    manifest["preparation_seconds"] = time.monotonic() - started\n    write_json(destination / "manifest.json", manifest)\n    write_json(destination / "resolved_config.json", {**config, "model_revision": model_sha, "dataset_revision": data_sha})\n    return manifest\n\n\ndef accelerator_imports(cpu_test=False):\n    global jax, jnp, optax, struct, serialization, traverse_util\n    os.environ["JAX_PLATFORMS"] = "cpu" if cpu_test else "tpu"\n    import jax\n    import jax.numpy as jnp\n    import optax\n    from flax import struct, serialization, traverse_util\n    return jax\n\n\ndef preflight(cpu_test=False):\n    accelerator_imports(cpu_test)\n    devices = jax.devices()\n    if len(devices) != 1:\n        raise RuntimeError(f"This notebook requires exactly ONE v5e chip; found {len(devices)} devices: {devices}. Select the v5e-1 TPU runtime in Colab.")\n    device = devices[0]\n    if not cpu_test:\n        kind = device.device_kind.lower().replace(" ", "")\n        if device.platform != "tpu" or not any(label in kind for label in ("v5lite", "v5e")):\n            raise RuntimeError(f"Expected a v5e-1 TPU, found {device}. Change the Colab runtime hardware accelerator.")\n    return {"platform": device.platform, "device_kind": device.device_kind,\n            "device_count": len(devices), "cpu_test": cpu_test,\n            "python": sys.version, "jax": jax.__version__,\n            "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()}}\n\n\ndef numpy_leaf(value):\n    if isinstance(value, jax.Array):\n        return np.asarray(jax.device_get(value))\n    return np.asarray(value)\n\n\ndef archive_tree(state_dict):\n    leaves = []\n    def encode(node):\n        if isinstance(node, dict):\n            return {key: encode(value) for key, value in node.items()}\n        name = f"arrays/{len(leaves):05d}.npy"\n        leaves.append((name, node))\n        return {"__array__": name}\n    return encode(state_dict), leaves\n\n\ndef save_archive(path, state, metadata):\n    """Write one array at a time; never materialize a full host checkpoint copy."""\n    tree, leaves = archive_tree(serialization.to_state_dict(state))\n    temporary = Path(str(path) + ".partial")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:\n        for name, leaf in leaves:\n            array = numpy_leaf(leaf)\n            with archive.open(name, "w", force_zip64=True) as stream:\n                np.lib.format.write_array(stream, array, allow_pickle=False)\n            del array\n        archive.writestr("checkpoint.json", json.dumps({"tree": tree, "metadata": metadata}, allow_nan=False))\n    temporary.replace(path)\n    return len(leaves)\n\n\ndef restore_archive(path, abstract_state):\n    """Restore arrays directly to the TPU before reconstructing the state class."""\n    with zipfile.ZipFile(path) as archive:\n        description = json.loads(archive.read("checkpoint.json"))\n        def decode(node):\n            if set(node) == {"__array__"}:\n                with archive.open(node["__array__"]) as stream:\n                    array = np.lib.format.read_array(stream, allow_pickle=False)\n                value = jax.device_put(array)\n                jax.block_until_ready(value)\n                return value\n            return {key: decode(value) for key, value in node.items()}\n        restored_dict = decode(description["tree"])\n    return serialization.from_state_dict(abstract_state, restored_dict)\n\n\nMODEL_HELPERS = \'@struct.dataclass\\nclass PilotState:\\n    step: jax.Array\\n    params: object\\n    opt_state: object\\n    rng: jax.Array\\n    data_position: jax.Array\\n\\n\\ndef make_model(cpu_fixture=False):\\n    # Kauldron\\\'s dtype registry assumes Linux\\\'s extended-precision NumPy alias.\\n    # This compatibility alias is used only by the explicitly marked Mac CPU fixture.\\n    if cpu_fixture and not hasattr(np, "float128"):\\n        np.float128 = np.longdouble\\n    from gemma import gm\\n    model = gm.nn.Gemma3_270M(dtype=jnp.float32)\\n    if cpu_fixture:\\n        config = dataclasses.replace(model.config, num_embed=128, embed_dim=32,\\n            hidden_dim=64, num_heads=2, head_dim=16, num_kv_heads=1,\\n            attention_types=model.config.attention_types[:2], sliding_window_size=8)\\n        model = model.clone(config=config)\\n    return model\\n\\n\\ndef make_optimizer(config, params):\\n    schedule = optax.join_schedules([\\n        optax.linear_schedule(0.0, config["learning_rate"], config["warmup_updates"]),\\n        optax.constant_schedule(config["learning_rate"]),\\n    ], [config["warmup_updates"]])\\n    mask = jax.tree.map(lambda value: value.ndim > 1, params)\\n    optimizer = optax.chain(optax.clip_by_global_norm(config["gradient_clip"]),\\n        optax.adamw(schedule, b1=config["adam_b1"], b2=config["adam_b2"],\\n                    eps=config["adam_eps"], weight_decay=config["weight_decay"], mask=mask))\\n    return optimizer, schedule\\n\'\n\ndef load_model_helpers(cpu_test=False):\n    accelerator_imports(cpu_test)\n    exec(MODEL_HELPERS, globals())\n\n\nPACKING_VERSION = "V2-best-fit-intact-chunks-1"\nINPUT_KEYS = ("tokens", "targets", "mask", "positions", "segment_ids")\n\n\ndef split_identity(documents):\n    return hashlib.sha256(("\\n".join(d["sha256"] for d in documents) + "\\n").encode()).hexdigest()\n\n\ndef pack_chunks(arrays, seed):\n    """Best-fit decreasing, preserving each original chunk\'s complete context."""\n    tokens, targets, masks = (arrays[key] for key in ("tokens", "targets", "mask"))\n    count, length = tokens.shape\n    lengths = masks.sum(axis=1).astype(int)\n    if np.any(lengths <= 0) or not np.array_equal(masks, np.arange(length)[None, :] < lengths[:, None]):\n        raise ValueError("Input chunks must have a nonempty contiguous prefix of scored targets.")\n    shuffled = np.random.default_rng(seed).permutation(count)\n    ordered = shuffled[np.argsort(-lengths[shuffled], kind="stable")]\n    buckets = [[] for _ in range(length + 1)]\n    bins, assignments = [], []\n    for source in ordered:\n        size = int(lengths[source])\n        free = next((room for room in range(size, length + 1) if buckets[room]), None)\n        if free is None:\n            index = len(bins)\n            bins.append(0)\n            free = length\n        else:\n            index = buckets[free].pop()\n        start = bins[index]\n        bins[index] += size\n        buckets[free - size].append(index)\n        assignments.append([int(source), index, start, size])\n    shape = (len(bins), length)\n    packed = {"tokens": np.zeros(shape, np.int32), "targets": np.zeros(shape, np.int32),\n              "mask": np.zeros(shape, np.bool_), "positions": np.zeros(shape, np.int32),\n              "segment_ids": np.full(shape, -1, np.int32)}\n    seen = np.zeros(count, np.int32)\n    for source, index, start, size in assignments:\n        sl = slice(start, start + size)\n        packed["tokens"][index, sl] = tokens[source, :size]\n        packed["targets"][index, sl] = targets[source, :size]\n        packed["mask"][index, sl] = True\n        packed["positions"][index, sl] = np.arange(size)\n        packed["segment_ids"][index, sl] = source\n        seen[source] += 1\n    # Verify content and positions before discarding the unpacked training arrays.\n    for source, index, start, size in assignments:\n        sl = slice(start, start + size)\n        if not (np.array_equal(packed["tokens"][index, sl], tokens[source, :size])\n                and np.array_equal(packed["targets"][index, sl], targets[source, :size])\n                and np.array_equal(packed["positions"][index, sl], np.arange(size))):\n            raise ValueError("Packing changed a source chunk.")\n    if not np.all(seen == 1) or int(packed["mask"].sum()) != int(lengths.sum()):\n        raise ValueError("Packing duplicated or dropped source targets.")\n    report = {"version": PACKING_VERSION, "source_chunks": count, "packed_sequences": len(bins),\n              "scored_tokens": int(lengths.sum()), "positions": int(np.prod(shape)),\n              "occupancy": float(lengths.sum() / np.prod(shape)),\n              "targets_preserved": True, "context_preserved": True,\n              "assignments_source_bin_start_length": assignments}\n    return packed, report\n\n\ndef verify_prepared(directory, baseline, cpu_test=False):\n    directory = Path(directory)\n    manifest = read_json(directory / "manifest.json")\n    for name, digest in manifest["files"].items():\n        if sha256_file(directory / name) != digest:\n            raise ValueError(f"Cached data failed its checksum: {name}")\n    split_hashes = {name: split_identity(split["documents"]) for name, split in manifest["splits"].items()}\n    if not cpu_test and split_hashes != baseline["split_hashes"]:\n        raise ValueError("Selected documents differ from the measured V1 data.")\n    train_hashes = {d["sha256"] for d in manifest["splits"]["train"]["documents"]}\n    valid_hashes = {d["sha256"] for d in manifest["splits"]["validation"]["documents"]}\n    if train_hashes & valid_hashes:\n        raise ValueError("Training and validation documents overlap.")\n    packing = read_json(directory / "packing.json")\n    if packing["version"] != PACKING_VERSION or not all(packing[k] for k in ("targets_preserved", "context_preserved")):\n        raise ValueError("Invalid packing record.")\n    return manifest, packing, split_hashes\n\n\ndef finalize_packing(directory, config):\n    directory = Path(directory)\n    with np.load(directory / "train.npz", allow_pickle=False) as archive:\n        original = {key: archive[key] for key in ("tokens", "targets", "mask")}\n    packed, report = pack_chunks(original, config["seed"])\n    np.savez_compressed(directory / "train.npz", **packed)\n    write_json(directory / "packing.json", report)\n    manifest = read_json(directory / "manifest.json")\n    manifest["packing_version"] = PACKING_VERSION\n    for name in ("train.npz", "validation.npz", "packing.json", "train.jsonl", "validation.jsonl"):\n        if (directory / name).exists():\n            manifest["files"][name] = sha256_file(directory / name)\n    write_json(directory / "manifest.json", manifest)\n    return report\n\n\ndef prepare_cached(config, work, output):\n    started = time.monotonic()\n    baseline = read_json(Path(work) / "baseline_v1.json")\n    identity = {key: config[key] for key in ("model_id", "model_revision", "dataset_id", "dataset_config",\n                "dataset_revision", "train_documents", "validation_documents", "sequence_length", "seed")}\n    identity.update(packing=PACKING_VERSION, document_hashes=baseline["split_hashes"])\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]\n    local_cache = Path(work).parent / "data-cache" / key\n    persistent_cache = Path(output).parent.parent / "V2-cache" / "data" / key\n    prepared = Path(work) / "prepared"\n    if prepared.exists():\n        raise FileExistsError("This run already has prepared data. Continue to training.")\n    reused = False\n    if local_cache.exists():\n        verify_prepared(local_cache, baseline)\n        reused = True\n    elif persistent_cache.exists():\n        verify_prepared(persistent_cache, baseline)\n        local_cache.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copytree(persistent_cache, local_cache)\n        reused = True\n    else:\n        local_cache.parent.mkdir(parents=True, exist_ok=True)\n        staging = Path(str(local_cache) + ".partial")\n        prepare(config, staging)\n        finalize_packing(staging, config)\n        verify_prepared(staging, baseline)\n        staging.rename(local_cache)\n        persistent_cache.parent.mkdir(parents=True, exist_ok=True)\n        persistent_staging = Path(str(persistent_cache) + ".partial")\n        shutil.copytree(local_cache, persistent_staging)\n        persistent_staging.rename(persistent_cache)\n    shutil.copytree(local_cache, prepared)\n    # Runtime timings belong to this run, never the run which populated the cache.\n    write_json(prepared / "resolved_config.json", config)\n    manifest, packing, _ = verify_prepared(prepared, baseline)\n    write_json(output / "preparation.json", {"cache_key": key, "cache_reused": reused,\n               "preparation_seconds": time.monotonic() - started,\n               "packed_sequences": packing["packed_sequences"], "occupancy": packing["occupancy"]})\n    print(f"Prepared {packing[\'packed_sequences\']:,} packed sequences; occupancy={packing[\'occupancy\']:.2%}; cache reused={reused}", flush=True)\n\n\ndef packed_attention_mask(segment_ids):\n    length = segment_ids.shape[-1]\n    same_segment = segment_ids[:, :, None] == segment_ids[:, None, :]\n    causal = jnp.arange(length)[:, None] >= jnp.arange(length)[None, :]\n    valid = segment_ids >= 0\n    allowed = same_segment & causal[None, :, :] & valid[:, :, None] & valid[:, None, :]\n    # Padding queries attend to themselves so every softmax has at least one key.\n    return allowed | ((~valid)[:, :, None] & jnp.eye(length, dtype=jnp.bool_)[None, :, :])\n\n\ndef loss_parts(model, params, batch):\n    compute_params = jax.tree.map(lambda x: x.astype(jnp.bfloat16), params)\n    arguments = {"tokens": batch["tokens"]}\n    if "segment_ids" in batch:\n        arguments.update(positions=batch["positions"], attention_mask=packed_attention_mask(batch["segment_ids"]))\n    logits = model.apply({"params": compute_params}, **arguments).logits.astype(jnp.float32)\n    selected = jnp.take_along_axis(logits, batch["targets"][..., None], axis=-1)[..., 0]\n    losses = jax.scipy.special.logsumexp(logits, axis=-1) - selected\n    weights = batch["mask"].astype(jnp.float32)\n    return jnp.sum(jnp.where(batch["mask"], losses, 0.0)), jnp.sum(weights)\n\n\ndef batch_training_arrays(arrays, batch_size, seed):\n    order = np.random.default_rng(seed).permutation(len(arrays["tokens"]))\n    padded = (-len(order)) % batch_size\n    result = {}\n    for key, values in arrays.items():\n        values = values[order]\n        if padded:\n            values = np.pad(values, ((0, padded), (0, 0)), constant_values=-1 if key == "segment_ids" else 0)\n        result[key] = values.reshape((-1, batch_size, values.shape[-1]))\n    return result, hashlib.sha256(order.astype("<i8").tobytes()).hexdigest()\n\n\ndef make_training_functions(model, optimizer):\n    def one_step(state, batch):\n        def objective(params):\n            total, count = loss_parts(model, params, batch)\n            return total / jnp.maximum(count, 1), count\n        (loss, count), grads = jax.value_and_grad(objective, has_aux=True)(state.params)\n        norm = optax.global_norm(grads)\n        finite = jnp.isfinite(loss) & jnp.isfinite(norm) & (count > 0)\n        def apply(_):\n            updates, new_opt = optimizer.update(grads, state.opt_state, state.params)\n            return state.replace(step=state.step + 1, params=optax.apply_updates(state.params, updates),\n                                 opt_state=new_opt, data_position=state.data_position + 1)\n        state = jax.lax.cond(finite, apply, lambda _: state, operand=None)\n        return state, (loss, count, norm, finite)\n\n    def run_group(state, training, requested_steps):\n        initial = (state, jnp.array(0, jnp.int32), jnp.array(0., jnp.float32),\n                   jnp.array(0, jnp.int32), jnp.array(0., jnp.float32), jnp.array(True))\n        def condition(carry):\n            return (carry[1] < requested_steps) & carry[5]\n        def body(carry):\n            state, steps, nll, tokens, max_norm, finite = carry\n            index = state.data_position % training["tokens"].shape[0]\n            batch = jax.tree.map(lambda values: jax.lax.dynamic_index_in_dim(values, index, keepdims=False), training)\n            state, (loss, count, norm, good) = one_step(state, batch)\n            return (state, steps + good.astype(jnp.int32), nll + loss * count,\n                    tokens + jnp.where(good, count, 0).astype(jnp.int32), jnp.maximum(max_norm, norm), finite & good)\n        state, steps, nll, tokens, norm, finite = jax.lax.while_loop(condition, body, initial)\n        return state, {"updates": steps, "nll_sum": nll, "tokens": tokens,\n                       "max_gradient_norm": norm, "finite": finite}\n    return one_step, jax.jit(run_group, donate_argnums=(0,))\n\n\ndef train_pilot(work, output):\n    work, output = Path(work), Path(output)\n    # An exclusive marker prevents a notebook-cell rerun from paying for another run.\n    with (work / "training.started").open("x") as marker:\n        marker.write("V2: one training attempt; automatic retry disabled.\\n")\n    environment = preflight()\n    load_model_helpers()\n    config = read_json(work / "prepared" / "resolved_config.json")\n    baseline = read_json(work / "baseline_v1.json")\n    data_dir = work / "prepared"\n    manifest, packing, split_hashes = verify_prepared(data_dir, baseline)\n    manifest_hash = sha256_file(data_dir / "manifest.json")\n    if shutil.disk_usage(work).free < 4 * 1024**3:\n        raise RuntimeError("V2 requires at least 4 GiB of free local disk for the full checkpoint.")\n    with np.load(data_dir / "train.npz", allow_pickle=False) as archive:\n        training_host = {key: archive[key] for key in INPUT_KEYS}\n    training_host, order_hash = batch_training_arrays(training_host, config["local_batch_per_chip"], config["seed"])\n    with np.load(data_dir / "validation.npz", allow_pickle=False) as archive:\n        validation_host = {key: archive[key] for key in ("tokens", "targets", "mask")}\n    if int(validation_host["mask"].sum()) != baseline["final_eval"]["scored_tokens"]:\n        raise ValueError("Validation target count differs from V1.")\n    if int(training_host["mask"].sum()) != packing["scored_tokens"]:\n        raise ValueError("Batch construction changed the target count.")\n    write_json(output / "config.json", config)\n    write_json(output / "environment.json", environment)\n    transfer_start = time.monotonic()\n    training = jax.device_put(training_host)\n    validation = jax.device_put(validation_host)\n    jax.block_until_ready((training, validation))\n    resident_inputs = all(jax.devices()[0] in leaf.devices() for leaf in jax.tree.leaves((training, validation)))\n    input_transfer_seconds = time.monotonic() - transfer_start\n    input_bytes = sum(x.nbytes for x in jax.tree.leaves(training_host))\n    del training_host, validation_host\n    print(f"Inputs on TPU: {input_bytes / 1024**2:.1f} MiB of packed training data.", flush=True)\n    probe_segments = jnp.array([[0, 0, 1, 1, -1]], jnp.int32)\n    probe_mask = np.asarray(jax.jit(packed_attention_mask)(probe_segments))[0]\n    expected_mask = np.array([[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0],\n                              [0, 0, 1, 1, 0], [0, 0, 0, 0, 1]], np.bool_)\n    isolated_segments = np.array_equal(probe_mask, expected_mask)\n    if not isolated_segments:\n        raise ValueError("Packed attention mask failed causal/segment/padding verification.")\n\n    model = make_model()\n    initialization_start = time.monotonic()\n    params = jax.jit(lambda key: model.init(key, tokens=jnp.ones((1, 1), jnp.int32))["params"])(jax.random.PRNGKey(config["seed"]))\n    jax.block_until_ready(params)\n    flat = traverse_util.flatten_dict(params)\n    parameter_count = sum(x.size for x in flat.values())\n    if parameter_count != 268_098_176 or not all(x.dtype == jnp.float32 for x in flat.values()):\n        raise ValueError("The native parameter count or FP32 master-weight requirement failed.")\n    embedding_shape = (262144, 640)\n    tied = (flat[("embedder", "input_embedding")].shape == embedding_shape\n            and sum(x.shape == embedding_shape for x in flat.values()) == 1)\n    probe_path = next(path for path, value in sorted(flat.items()) if path[0].startswith("layer_") and value.ndim > 1)\n    def probe(parameters):\n        return np.array(jax.device_get(traverse_util.flatten_dict(parameters)[probe_path].reshape(-1)[:64]))\n    initial_probe = probe(params)\n    optimizer, schedule = make_optimizer(config, params)\n    state = PilotState(jnp.array(0, jnp.int32), params, optimizer.init(params),\n                       jax.random.PRNGKey(config["seed"]), jnp.array(0, jnp.int32))\n    del flat, params\n    jax.block_until_ready(state)\n    initialization_seconds = time.monotonic() - initialization_start\n    _, group_jit = make_training_functions(model, optimizer)\n\n    def evaluation_all(parameters, arrays):\n        def body(index, totals):\n            batch = jax.tree.map(lambda value: jax.lax.dynamic_slice_in_dim(value, index, 1), arrays)\n            nll, count = loss_parts(model, parameters, batch)\n            # Float64 is intentionally unnecessary: report FP32 accumulation explicitly.\n            return totals[0] + nll, totals[1] + count\n        return jax.lax.fori_loop(0, arrays["tokens"].shape[0], body,\n                                (jnp.array(0., jnp.float32), jnp.array(0., jnp.float32)))\n    evaluation_jit = jax.jit(evaluation_all)\n    print("Compiling one dynamic-count training loop and the validation loop; zero weight updates...", flush=True)\n    compilation_start = time.monotonic()\n    executable = group_jit.lower(state, training, jnp.array(1, jnp.int32)).compile()\n    evaluator = evaluation_jit.lower(state.params, validation).compile()\n    compilation_seconds = time.monotonic() - compilation_start\n    if int(state.step) != 0:\n        raise ValueError("An optimizer update occurred before timing.")\n    try:\n        analysis = executable.memory_analysis()\n        compiled_memory = {key: int(getattr(analysis, key)) for key in\n            ("argument_size_in_bytes", "output_size_in_bytes", "alias_size_in_bytes", "temp_size_in_bytes")}\n    except (AttributeError, TypeError, NotImplementedError):\n        compiled_memory = None\n\n    def evaluate(parameters):\n        nll, tokens = jax.block_until_ready(evaluator(parameters, validation))\n        nll, tokens = float(nll), int(tokens)\n        if tokens != baseline["final_eval"]["scored_tokens"] or not math.isfinite(nll):\n            raise ValueError("Invalid validation loss or target count.")\n        return {"loss": nll / tokens, "nll_sum": nll, "scored_tokens": tokens,\n                "document_hash": split_hashes["validation"], "accumulation_dtype": "float32"}\n\n    print("Initial validation...", flush=True)\n    phase = time.monotonic()\n    before = evaluate(state.params)\n    initial_eval_seconds = time.monotonic() - phase\n    jax.block_until_ready(state)\n    print("Starting the ONE 60-second V2 training run...", flush=True)\n    log_rows = []\n    updates = tokens_processed = 0\n    max_seconds_per_update = 0.0\n    last_progress = last_start = 0.0\n    started = time.monotonic()\n    while True:\n        elapsed = time.monotonic() - started\n        remaining = config["training_seconds"] - elapsed\n        if remaining <= 0:\n            break\n        # First dispatch is one timed update. Near the deadline, return to one update.\n        group_size = config["updates_per_dispatch"]\n        threshold = max(2.0, 2 * group_size * max_seconds_per_update)\n        requested = group_size if updates > 0 and remaining > threshold else 1\n        last_start = time.monotonic() - started\n        if last_start >= config["training_seconds"]:\n            break\n        call_start = time.monotonic()\n        state, metrics = jax.block_until_ready(executable(state, training, jnp.array(requested, jnp.int32)))\n        metrics = {key: np.asarray(value).item() for key, value in metrics.items()}\n        if not metrics["finite"] or metrics["updates"] != requested:\n            raise FloatingPointError("Nonfinite training value; affected update rejected; no retry.")\n        completed = int(metrics["updates"])\n        updates += completed\n        tokens_processed += int(metrics["tokens"])\n        duration = time.monotonic() - call_start\n        max_seconds_per_update = max(max_seconds_per_update, duration / completed)\n        end_seconds = time.monotonic() - started\n        log_rows.append({**metrics, "update": updates, "requested_updates": requested,\n                         "start_seconds": last_start, "end_seconds": end_seconds,\n                         "loss": metrics["nll_sum"] / metrics["tokens"]})\n        if len(log_rows) == 1 or end_seconds - last_progress >= 2:\n            print(f"updates={updates:,} tokens={tokens_processed:,} loss={log_rows[-1][\'loss\']:.5f} elapsed={end_seconds:.2f}s", flush=True)\n            last_progress = end_seconds\n    jax.block_until_ready(state)\n    training_seconds = time.monotonic() - started\n    # Disk writes happen after timing; no per-update transfers or log flushes.\n    with (output / "training.jsonl").open("w") as log:\n        for row in log_rows:\n            log.write(json.dumps(row, allow_nan=False) + "\\n")\n    changed = not np.array_equal(initial_probe, probe(state.params))\n    if int(state.step) != updates or int(state.data_position) != updates:\n        raise ValueError("Optimizer count and data cursor differ from completed updates.")\n    phase = time.monotonic()\n    after = evaluate(state.params)\n    final_eval_seconds = time.monotonic() - phase\n    memory_stats = jax.devices()[0].memory_stats()\n    checkpoint_start = time.monotonic()\n    metadata = {"version": "V2", "config": config, "parameter_count": parameter_count,\n                "model_initialization": "random", "model_implementation": "gemma==4.0.1; gm.nn.Gemma3_270M",\n                "data_manifest_sha256": manifest_hash, "packing_version": PACKING_VERSION,\n                "data_order_sha256": order_hash, "data_batches": training["tokens"].shape[0],\n                "data_position_unit": "completed batches; fixed seeded order repeats after a complete pass",\n                "updates": updates, "checkpoint_fields": ["step", "params", "opt_state", "rng", "data_position"]}\n    print("Saving one full resumable checkpoint (approximately 3.2 GB)...", flush=True)\n    local_checkpoint = work / "checkpoint.npz"\n    leaves = save_archive(local_checkpoint, state, metadata)\n    digest = sha256_file(local_checkpoint)\n    checkpoint = output / "checkpoint.npz"\n    partial = output / "checkpoint.npz.partial"\n    shutil.copyfile(local_checkpoint, partial)\n    if sha256_file(partial) != digest:\n        raise ValueError("Persistent checkpoint checksum differs from the local file.")\n    partial.rename(checkpoint)\n    with zipfile.ZipFile(checkpoint) as archive:\n        saved = json.loads(archive.read("checkpoint.json"))\n        tree = saved["tree"]\n        if set(tree) != {"step", "params", "opt_state", "rng", "data_position"} or saved["metadata"] != metadata:\n            raise ValueError("Checkpoint fields or metadata differ from the final state.")\n        if sum(name.startswith("arrays/") for name in archive.namelist()) != leaves:\n            raise ValueError("Checkpoint array count differs.")\n    write_json(output / "checkpoint_manifest.json", {"sha256": digest, "bytes": checkpoint.stat().st_size,\n                "array_leaves": leaves, "verification": "local/persistent SHA256 equality and state-schema inspection"})\n    checkpoint_seconds = time.monotonic() - checkpoint_start\n    for name in ("manifest.json", "packing.json"):\n        shutil.copyfile(data_dir / name, output / ("data_manifest.json" if name == "manifest.json" else name))\n    import gzip\n    with gzip.open(output / "source_documents.jsonl.gz", "wb") as stream:\n        for name in ("validation.jsonl", "train.jsonl"):\n            with (data_dir / name).open("rb") as source:\n                shutil.copyfileobj(source, stream)\n    runtime_elapsed = time.monotonic() - config["notebook_start_monotonic"]\n    rate = config["compute_units_per_hour"]\n    throughput = tokens_processed / training_seconds\n    occupancy = tokens_processed / (updates * config["local_batch_per_chip"] * config["sequence_length"])\n    checks = {\n        "V2-001_ONE_RUN": True,\n        "V2-002_ONE_V5E": environment["platform"] == "tpu" and environment["device_count"] == 1,\n        "V2-003_TARGETS_PRESERVED": packing["targets_preserved"],\n        "V2-004_ISOLATED_SEGMENTS": isolated_segments,\n        "V2-005_CONTEXT_PRESERVED": packing["context_preserved"],\n        "V2-006_RESIDENT_INPUTS": resident_inputs,\n        "V2-007_GROUP_SIZE": all(1 <= row["requested_updates"] <= 16 for row in log_rows),\n        "V2-008_TIMING": training_seconds >= config["training_seconds"] and last_start < config["training_seconds"],\n        "V2-009_FINITE": all(row["finite"] for row in log_rows),\n        "V2-010_VALIDATION": before["document_hash"] == after["document_hash"] and before["scored_tokens"] == after["scored_tokens"],\n        "V2-011_CHECKPOINT": checkpoint.exists(),\n        "V2-012_TIMINGS": all(value >= 0 for value in (initialization_seconds, compilation_seconds,\n                                  input_transfer_seconds, initial_eval_seconds, final_eval_seconds, checkpoint_seconds)),\n        "V2-013_NATIVE_MODEL": parameter_count == 268_098_176 and tied,\n        "V2-014_ZERO_UNTIMED_UPDATES": int(state.step) == updates,\n        "V2-015_TRANSFORMER_CHANGED": changed,\n        "V2-016_MINIMUM_UPDATES": updates >= config["minimum_updates"],\n    }\n    result = {"version": "V2", "cpu_test": False, "run_count": 1, "config": config,\n        "device": environment["device_kind"], "model_parameters": parameter_count,\n        "input_transfer_seconds": input_transfer_seconds, "resident_training_bytes": input_bytes,\n        "initialization_seconds": initialization_seconds, "compilation_seconds": compilation_seconds,\n        "initial_eval_seconds": initial_eval_seconds, "final_eval_seconds": final_eval_seconds,\n        "training_seconds": training_seconds, "training_overrun_seconds": training_seconds - config["training_seconds"],\n        "last_dispatch_start_seconds": last_start, "host_dispatches": len(log_rows),\n        "updates": updates, "scored_tokens": tokens_processed, "tokens_per_second": throughput,\n        "training_occupancy": occupancy, "prepared_occupancy": packing["occupancy"],\n        "data_passes": updates // training["tokens"].shape[0], "data_manifest_sha256": manifest_hash,\n        "initial_eval": before, "final_eval": after, "final_learning_rate": float(schedule(state.step)),\n        "checkpoint_save_verify_seconds": checkpoint_seconds, "checkpoint_bytes": checkpoint.stat().st_size,\n        "checkpoint_sha256": digest, "checkpoint_array_leaves": leaves,\n        "memory_stats": memory_stats, "compiled_memory_estimate": compiled_memory,\n        "memory_scope": "JAX allocator counters sampled after final validation; compiler estimate reported separately.",\n        "notebook_elapsed_seconds": runtime_elapsed,\n        "compute_units_estimate": {"rate_per_hour": rate, "training": training_seconds * rate / 3600,\n             "through_report": runtime_elapsed * rate / 3600, "overhead_through_report": (runtime_elapsed - training_seconds) * rate / 3600},\n        "cost_note": "Estimates at the user-supplied 2.92 CU/hour; Colab billing is authoritative. Runtime after report creation is additional.",\n        "v1_comparison": {"tokens_per_second": baseline["tokens_per_second"],\n             "speed_ratio": throughput / baseline["tokens_per_second"], "final_loss": baseline["final_eval"]["loss"],\n             "final_loss_difference": after["loss"] - baseline["final_eval"]["loss"], "source": baseline["source"]},\n        "performance_goals": {"occupancy_at_least_95_percent": occupancy >= .95,\n             "tokens_per_second_at_least_18000": throughput >= 18000,\n             "final_loss_at_most_v1": after["loss"] <= baseline["final_eval"]["loss"]},\n        "requirements": {key: "PASS" if value else "FAIL" for key, value in checks.items()},\n        "engineering_status": "PASS" if all(checks.values()) else "FAIL",\n        "learning_status": "PASS" if after["loss"] < before["loss"] else "FAIL"}\n    write_json(output / "result.json", result)\n    print(json.dumps({key: result[key] for key in ("engineering_status", "learning_status", "updates", "training_seconds", "tokens_per_second", "v1_comparison")}, indent=2), flush=True)\n    if not all(checks.values()):\n        raise RuntimeError("V2 engineering requirement failed. Saved result.json contains the details.")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--phase", choices=["preflight", "prepare", "train"], required=True)\n    parser.add_argument("--work", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    try:\n        if args.phase == "preflight":\n            info = preflight()\n            write_json(args.output / "preflight.json", info)\n            print(json.dumps({key: info[key] for key in ("platform", "device_kind", "device_count", "jax")}, indent=2))\n        elif args.phase == "prepare":\n            prepare_cached(read_json(args.work / "config.json"), args.work, args.output)\n        else:\n            cache = Path(args.work).parent / "jax-cache-V2"\n            cache.mkdir(parents=True, exist_ok=True)\n            os.environ["JAX_COMPILATION_CACHE_DIR"] = str(cache)\n            os.environ["JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS"] = "1"\n            try:\n                train_pilot(args.work, args.output)\n            finally:\n                # Persist cache once after execution; all live compilation uses local storage.\n                persistent = args.output.parent.parent / "V2-cache" / "jax"\n                try:\n                    persistent.mkdir(parents=True, exist_ok=True)\n                    shutil.copytree(cache, persistent, dirs_exist_ok=True)\n                except OSError as error:\n                    write_json(args.output / "cache_warning.json", {"warning": str(error), "training_retried": False})\n    except Exception:\n        write_json(args.output / "failure.json", {"phase": args.phase, "error": traceback.format_exc(), "automatic_retry": False})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'

BASELINE_V1 = {'model_parameters': 268098176, 'training_seconds': 60.01888866399986, 'updates': 1271, 'scored_tokens': 684589, 'tokens_per_second': 11406.225860536897, 'initial_eval': {'loss': 12.490467064324235, 'manifest_sha256': 'c93a5c26feae779aa34eb74a2701fa89a0663ce7d5f1090a6743dc61fa203f55', 'nll_sum': 165236.3887939453, 'scored_tokens': 13229}, 'final_eval': {'loss': 6.872893265166136, 'manifest_sha256': 'c93a5c26feae779aa34eb74a2701fa89a0663ce7d5f1090a6743dc61fa203f55', 'nll_sum': 90921.50500488281, 'scored_tokens': 13229}, 'notebook_elapsed_seconds': 343.2592167699995, 'split_hashes': {'train': 'a1152a578b44071931157992cec95f047f6c037c992843b71315a89f480f1867', 'validation': '09c6417709b313ce9b0464836f057eaaa8d8b11edcca72c9a6434be608106608'}, 'training_occupancy': 0.5259983834087333, 'source': 'User-provided gemma_colab_pilot_reports; measured V1 results, not a V2 measurement.'}
CONFIG = {'experiment': 'GEM-COLAB-V2', 'version': 'V2', 'model_id': 'google/gemma-3-270m', 'model_revision': '9b0cfec892e2bc2afd938c98eabe4e4a7b1e0ca1', 'dataset_id': 'HuggingFaceFW/fineweb', 'dataset_config': 'sample-10BT', 'dataset_revision': '9bb295ddab0e05d785b879661af7260fed5140fc', 'train_documents': 10000, 'validation_documents': 32, 'sequence_length': 1024, 'seed': 42, 'expected_devices': 1, 'local_batch_per_chip': 2, 'updates_per_dispatch': 16, 'training_seconds': 60.0, 'minimum_updates': 5, 'learning_rate': 0.0003, 'warmup_updates': 5, 'adam_b1': 0.9, 'adam_b2': 0.999, 'adam_eps': 1e-08, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'compute_units_per_hour': 2.92}

CONFIG["notebook_start_monotonic"] = NOTEBOOK_START_MONOTONIC
CONFIG["setup_seconds"] = SETUP_SECONDS
PROGRAM_PATH = WORK / "pilot_V2.py"
PROGRAM_PATH.write_text(PROGRAM_SOURCE)
(WORK / "baseline_v1.json").write_text(json.dumps(BASELINE_V1, indent=2))
(WORK / "config.json").write_text(json.dumps(CONFIG, indent=2))
shutil.copyfile(PROGRAM_PATH, OUTPUT / "pilot_source.py")
shutil.copyfile(WORK / "baseline_v1.json", OUTPUT / "baseline_v1.json")
shutil.copyfile(WORK / "config.json", OUTPUT / "requested_config.json")
shutil.copyfile(LOCK_PATH, OUTPUT / "requirements.lock")

def run_phase(phase, token=None):
    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.pop("HF_TOKEN", None)
    if token:
        environment["HF_TOKEN"] = token
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["PYTHONUNBUFFERED"] = "1"
    environment["MPLCONFIGDIR"] = str(BASE / "matplotlib-cache")
    command = [str(PYTHON), str(PROGRAM_PATH), "--phase", phase,
               "--work", str(WORK), "--output", str(OUTPUT)]
    # Stream the subprocess output to the notebook. Logs are buffered to reduce Drive writes.
    with (OUTPUT / (phase + ".log")).open("w") as log:
        process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"V2 {phase} failed. See {OUTPUT / (phase + '.log')} and failure.json. No automatic retry was attempted.")


## 4. Verify the accelerator
This stops before dataset download if the runtime is not exactly one v5e TPU. It performs no training.


In [ ]:
run_phase("preflight")


## 5. Prepare and pack the fixed data
Best-fit packing keeps each original document chunk intact. Positions restart for each segment. Compact segment IDs are stored; full attention masks are constructed per training batch. Cached files are reused only after checksum and document-identity verification.


In [ ]:
PREPARATION_START = time.monotonic()
run_phase("prepare", token=HF_TOKEN)
HF_TOKEN = None
PREPARATION_SECONDS = time.monotonic() - PREPARATION_START
preparation = json.loads((OUTPUT / "preparation.json").read_text())
print(json.dumps(preparation, indent=2))


## 6. Execute the single 60-second training run
This cell initializes, compiles, performs initial validation, trains once, performs final validation, and saves one checkpoint. Compilation performs zero optimizer updates. A dynamic counted JAX loop supports both 16-update calls and one-update calls without changing input shapes. Near the deadline it returns to one-update calls; completed-work time and any overrun are reported.


In [ ]:
run_phase("train")


## 7. Read the actual V2 result
All V2 values below come from your execution. V1 values are the previously measured reference.


In [ ]:
from IPython.display import display, Markdown
result = json.loads((OUTPUT / "result.json").read_text())
rows = [
    ("Engineering", result["engineering_status"]),
    ("Learning", result["learning_status"]),
    ("Completed training runs", result["run_count"]),
    ("Actual training seconds", f'{result["training_seconds"]:.3f}'),
    ("Training overrun seconds", f'{result["training_overrun_seconds"]:.3f}'),
    ("Completed optimizer updates", f'{result["updates"]:,}'),
    ("Host dispatches", result["host_dispatches"]),
    ("Useful training tokens", f'{result["scored_tokens"]:,}'),
    ("Useful position occupancy", f'{result["training_occupancy"]:.2%}'),
    ("Useful tokens/second", f'{result["tokens_per_second"]:,.1f}'),
    ("Throughput / measured V1", f'{result["v1_comparison"]["speed_ratio"]:.3f}x'),
    ("Initial validation loss", f'{result["initial_eval"]["loss"]:.6f}'),
    ("Final validation loss", f'{result["final_eval"]["loss"]:.6f}'),
    ("Final loss difference from V1", f'{result["v1_comparison"]["final_loss_difference"]:+.6f}'),
    ("Environment setup seconds", f'{SETUP_SECONDS:.1f}'),
    ("Preparation/cache seconds", f'{PREPARATION_SECONDS:.1f}'),
    ("Compilation seconds", f'{result["compilation_seconds"]:.1f}'),
    ("Checkpoint save/verification seconds", f'{result["checkpoint_save_verify_seconds"]:.1f}'),
    ("Checkpoint GB", f'{result["checkpoint_bytes"] / 1e9:.3f}'),
    ("Estimated training CU", f'{result["compute_units_estimate"]["training"]:.5f}'),
    ("Estimated CU through report", f'{result["compute_units_estimate"]["through_report"]:.5f}'),
]
display(Markdown("| Measurement | Result |\n|---|---|\n" + "\n".join(f"| {name} | {value} |" for name, value in rows)))
print("Requirements:", json.dumps(result["requirements"], indent=2))
print("Performance goals:", json.dumps(result["performance_goals"], indent=2))
print("All artifacts:", OUTPUT)
print("CU values are estimates at 2.92 CU/hour. Colab billing includes additional connected runtime.")


## 8. Download the report bundle
The checkpoint remains in Drive. The small bundle includes the result, logs, source, dependency lock, packing record, V1 reference, and checkpoint checksum manifest.


In [ ]:
import zipfile
from google.colab import files

REPORT_ZIP = WORK / "gemma_colab_pilot_V2_reports.zip"
with zipfile.ZipFile(REPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ["result.json", "environment.json", "preflight.json", "requested_config.json", "config.json",
                     "data_manifest.json", "packing.json", "preparation.json", "baseline_v1.json", "training.jsonl",
                     "checkpoint_manifest.json", "requirements.lock", "pilot_source.py", "preflight.log",
                     "prepare.log", "train.log", "cache_warning.json", "failure.json"]:
        path = OUTPUT / filename
        if path.is_file():
            bundle.write(path, filename)
shutil.copyfile(REPORT_ZIP, OUTPUT / REPORT_ZIP.name)
files.download(str(REPORT_ZIP))
print("Checkpoint:", OUTPUT / "checkpoint.npz")
print("Finished. Use Runtime > Disconnect and delete runtime when you are done.")


## Reproducibility and interpretation

- Model and dataset revisions are fixed to the commits used by your measured V1 run. The ordered document-hash identities must match V1 for both splits.
- Packing preserves original chunks, including their original context limits and EOS targets. A seeded permutation defines the training order; the final partial batch is padded rather than duplicating real sequences.
- The checkpoint includes parameters, AdamW moments/counts, PRNG state, completed-update count, batch cursor, configuration, packing version, data order hash, and source manifest hash. The embedded `restore_archive` routine reconstructs the full state from an abstract state template. The training entry point initializes randomly for this one pilot.
- Checkpoint persistence is verified by matching local and Drive SHA256 hashes plus schema inspection. V2 does not perform a second full-state download and third validation pass during the ordinary run. Restore execution has not been locally tested.
- Validation preserves the original V1 inputs and masks. The compiled validation loop accumulates totals in FP32; tiny numerical differences from V1's host accumulation are possible.
- A faster token rate and lower validation loss are separate outcomes. Performance goals are not included in the engineering PASS decision.
- JAX allocator counters and compiler memory estimates are reported separately. Neither is presented as an independently measured complete hardware-memory trace.
- Caches reduce repeated setup when valid matching artifacts exist. First-run compilation still has a cost. Compiler-cache reuse depends on matching executable signatures and runtime versions.
- A repeated training-cell click stops at the exclusive start marker. The notebook does not retry failures or select a different batch size.
- No local checks or tests were run for V2, at your request. All V2 runtime measurements remain pending your Colab execution.

Sources: [native Gemma](https://gemma-llm.readthedocs.io/en/latest/api/gm/nn/Gemma3_270M.html), [JAX control flow](https://docs.jax.dev/en/latest/control-flow.html), [JAX persistent cache](https://docs.jax.dev/en/latest/persistent_compilation_cache.html), [FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb), [NASA requirements guidance](https://www.nasa.gov/reference/appendix-c-how-to-write-a-good-requirement/).
